# Лабораторная работа №1: Классификация цвета автомобиля (DVM, фронтальные виды)

Цель: обучить и сравнить 3 классификатора цвета автомобиля на DVM:
1. `ScratchResNet` (написан вручную, обучение с нуля)
2. `ResNet18` (предобучен на ImageNet, дообучение)
3. `MobileNetV3-Large` (предобучен на ImageNet, дообучение)

Критерий: `F1_macro`, целевое требование — **больше 0.8**.


## Подход к данным (с учетом дисбаланса)

Чтобы оставить больше классов и при этом контролировать дисбаланс, используется схема:
- класс `unlisted` удаляется как шумовой;
- редкие классы **по умолчанию не склеиваются**, чтобы сохранить более детальную классификацию;
- для train применяется комбинация:
  - **undersampling частых классов** (ограничение сверху),
  - **WeightedRandomSampler** (оверсемплинг редких на уровне батчей).

При необходимости можно включить объединение редких классов в `other_rare` через `Config.merge_rare_into_other=True`.


In [ ]:
import os
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

try:
    import seaborn as sns
    HAS_SEABORN = True
except ModuleNotFoundError:
    sns = None
    HAS_SEABORN = False
    print('seaborn не найден: используем fallback на matplotlib')

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as tv_models

if HAS_SEABORN:
    sns.set_theme(style='whitegrid')
else:
    plt.style.use('ggplot')


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('CUDA available:', torch.cuda.is_available())


In [ ]:
@dataclass
class Config:
    data_root: str = 'auto'  # auto | явный путь | через DVM_DATA_ROOT

    # вычислительный профиль (CPU-friendly)
    image_size: int = 128
    batch_size: int = 64
    eval_batch_size: int = 256
    num_workers: int = 2

    # обработка классов (оставляем больше классов)
    drop_unlisted: bool = True
    merge_rare_into_other: bool = False
    rare_threshold: int = 60
    min_samples_per_class: int = 20

    # балансировка train
    max_train_samples_per_class: int = 2200

    # обучение
    epochs_scratch: int = 7
    epochs_pretrained_1: int = 5
    epochs_pretrained_2: int = 5
    lr_scratch: float = 1e-3
    lr_pretrained: float = 3e-4
    weight_decay: float = 1e-4

    # логирование
    use_tqdm: bool = True
    tqdm_leave: bool = False

cfg = Config()
cfg


## (Опционально) Скачать и распаковать DVM front-view прямо из ноутбука

Если датасет еще не скачан, выставьте `DOWNLOAD_DVM_FRONT = True` и запустите следующую ячейку один раз.
По умолчанию скачивание выключено, чтобы не тянуть ~730MB при каждом запуске.


In [ ]:
from pathlib import Path
import shutil
import urllib.request
import zipfile

DOWNLOAD_DVM_FRONT = False
DVM_FRONT_URL = 'https://ndownloader.figshare.com/files/34792480'  # Confirmed_fronts.zip (~730MB)
ZIP_PATH = Path('data/raw/Confirmed_fronts.zip')
EXTRACT_ROOT = Path('data/raw')
TARGET_DIR = EXTRACT_ROOT / 'dvm_front'


def download_file(url: str, dst: Path, chunk_size: int = 8 * 1024 * 1024):
    dst.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(url) as resp, open(dst, 'wb') as f:
        total = resp.headers.get('Content-Length')
        total = int(total) if total and total.isdigit() else None
        downloaded = 0

        while True:
            chunk = resp.read(chunk_size)
            if not chunk:
                break
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                print(f'\rСкачано {downloaded/1024/1024:.1f} / {total/1024/1024:.1f} MB', end='')
        if total:
            print()


def extract_dvm_zip(zip_path: Path, extract_root: Path, target_dir: Path):
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_root)

    candidate = extract_root / 'Confirmed_fronts'
    if candidate.exists():
        if target_dir.exists():
            shutil.rmtree(target_dir)
        candidate.rename(target_dir)

    if not target_dir.exists():
        raise RuntimeError('После распаковки не найден каталог data/raw/dvm_front')


if DOWNLOAD_DVM_FRONT:
    if TARGET_DIR.exists() and any(TARGET_DIR.rglob('*.jpg')):
        print('Датасет уже найден в', TARGET_DIR)
    else:
        if not ZIP_PATH.exists():
            print('Скачивание архива...')
            download_file(DVM_FRONT_URL, ZIP_PATH)
        else:
            print('Архив уже скачан:', ZIP_PATH)

        print('Распаковка...')
        extract_dvm_zip(ZIP_PATH, EXTRACT_ROOT, TARGET_DIR)
        print('Готово. Датасет распакован в', TARGET_DIR)
else:
    print('Скачивание отключено. Если нужно скачать датасет, выставьте DOWNLOAD_DVM_FRONT = True и запустите ячейку.')


In [ ]:
def canonical_color(raw: str) -> str:
    c = raw.strip().lower().replace('-', ' ')
    direct = {
        'grey': 'gray',
        'silver metallic': 'silver',
        'blue metallic': 'blue',
        'red metallic': 'red',
        'white pearl': 'white',
        'black metallic': 'black',
    }
    if c in direct:
        return direct[c]

    groups = {
        'black': ['black', 'jet black'],
        'white': ['white', 'ivory'],
        'gray': ['gray', 'grey', 'graphite', 'charcoal'],
        'silver': ['silver'],
        'blue': ['blue', 'navy', 'azure', 'teal'],
        'red': ['red', 'maroon', 'burgundy'],
        'green': ['green', 'lime', 'olive'],
        'yellow': ['yellow'],
        'orange': ['orange', 'amber'],
        'brown': ['brown', 'bronze', 'chocolate'],
        'purple': ['purple', 'violet', 'magenta'],
        'gold': ['gold'],
        'beige': ['beige', 'cream', 'sand'],
        'unlisted': ['unlisted'],
    }

    for key, tokens in groups.items():
        if any(tok in c for tok in tokens):
            return key

    return c


def extract_color_from_filename(filename: str):
    stem = Path(filename).stem
    parts = stem.split('$$')
    # DVM format: Brand$$Model$$Year$$Color$$...
    if len(parts) >= 4:
        return parts[3]
    return None


def resolve_data_root(config_value: str) -> Path:
    if config_value != 'auto':
        p = Path(config_value).expanduser().resolve()
        if p.exists():
            return p

    env_path = os.getenv('DVM_DATA_ROOT')
    if env_path:
        p = Path(env_path).expanduser().resolve()
        if p.exists():
            return p

    candidates = [
        Path.cwd() / 'data' / 'raw' / 'dvm_front',
        Path.cwd() / '..' / 'data' / 'raw' / 'dvm_front',
        Path(__file__).resolve().parent / 'data' / 'raw' / 'dvm_front' if '__file__' in globals() else None,
    ]

    for p in candidates:
        if p is not None and p.exists():
            return p.resolve()

    raise FileNotFoundError(
        'Не найден DVM датасет. Укажите путь через cfg.data_root или переменную DVM_DATA_ROOT.'
    )


def build_index(data_root: str) -> pd.DataFrame:
    root = resolve_data_root(data_root)

    rows = []
    valid_ext = {'.jpg', '.jpeg', '.png'}

    for img_path in root.rglob('*'):
        if img_path.suffix.lower() not in valid_ext:
            continue

        rel_parts = img_path.relative_to(root).parts
        raw_color = rel_parts[-2] if len(rel_parts) >= 5 else None

        parsed = extract_color_from_filename(img_path.name)
        if parsed:
            raw_color = parsed

        if not raw_color:
            continue

        rows.append({
            'path': str(img_path),
            'raw_color': raw_color,
            'color': canonical_color(raw_color),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError('Изображения не найдены')
    return df


def preprocess_df(df: pd.DataFrame, cfg: Config):
    df = df.copy()
    report = {}

    report['classes_before'] = sorted(df['color'].unique().tolist())

    if cfg.drop_unlisted:
        had_unlisted = 'unlisted' in set(df['color'].unique().tolist())
        df = df[df['color'] != 'unlisted'].copy()
        report['removed_unlisted_class'] = bool(had_unlisted)
    else:
        report['removed_unlisted_class'] = False

    report['merged_rare_classes'] = []
    if cfg.merge_rare_into_other:
        counts = df['color'].value_counts()
        rare = sorted(counts[counts < cfg.rare_threshold].index.tolist())
        if rare:
            df.loc[df['color'].isin(rare), 'color'] = 'other_rare'
        report['merged_rare_classes'] = rare

    counts2 = df['color'].value_counts()
    keep = set(counts2[counts2 >= cfg.min_samples_per_class].index.tolist())
    before_min_filter = set(df['color'].unique().tolist())
    df = df[df['color'].isin(keep)].copy()
    after_min_filter = set(df['color'].unique().tolist())

    report['removed_by_min_samples_classes'] = sorted(list(before_min_filter - after_min_filter))
    report['classes_after'] = sorted(df['color'].unique().tolist())

    df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
    return df, report


def make_splits(df: pd.DataFrame):
    train_df, temp_df = train_test_split(
        df,
        test_size=0.30,
        random_state=42,
        stratify=df['color'],
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=42,
        stratify=temp_df['color'],
    )
    return train_df, val_df, test_df


resolved_root = resolve_data_root(cfg.data_root)
print('Resolved data root:', resolved_root)

raw_df = build_index(str(resolved_root))
df, prep_report = preprocess_df(raw_df, cfg)

print('Raw images:', len(raw_df))
print('After preprocessing:', len(df))
print('Classes before preprocessing:', len(prep_report['classes_before']))
print('Classes after preprocessing:', len(prep_report['classes_after']))
print('Removed unlisted class:', prep_report['removed_unlisted_class'])
print('Merged rare classes:', prep_report['merged_rare_classes'])
print('Removed by min_samples_per_class:', prep_report['removed_by_min_samples_classes'])
print(df['color'].value_counts())


In [ ]:
plt.figure(figsize=(11, 4))
counts = df['color'].value_counts()
if HAS_SEABORN:
    sns.barplot(x=counts.index, y=counts.values)
else:
    plt.bar(counts.index, counts.values)
plt.xticks(rotation=45, ha='right')
plt.title('Распределение классов после preprocessing')
plt.ylabel('Количество')
plt.tight_layout()
plt.show()


In [ ]:
train_df, val_df, test_df = make_splits(df)

label2id = {c: i for i, c in enumerate(sorted(train_df['color'].unique()))}
id2label = {i: c for c, i in label2id.items()}

for part in (train_df, val_df, test_df):
    part['label'] = part['color'].map(label2id)

print('Split sizes:')
print('train:', len(train_df), 'val:', len(val_df), 'test:', len(test_df))
print('labels:', label2id)


In [ ]:
def cap_frequent_classes(frame: pd.DataFrame, max_per_class: int) -> pd.DataFrame:
    parts = []
    for _, grp in frame.groupby('color'):
        n = min(len(grp), max_per_class)
        parts.append(grp.sample(n=n, random_state=42))
    out = pd.concat(parts, ignore_index=True)
    return out.sample(frac=1.0, random_state=42).reset_index(drop=True)


train_balanced_df = cap_frequent_classes(train_df, cfg.max_train_samples_per_class)
print('Train original:', len(train_df))
print('Train after undersampling frequent classes:', len(train_balanced_df))
print(train_balanced_df['color'].value_counts())


In [ ]:
class ColorDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row['label'])


train_tfms = T.Compose([
    T.Resize((cfg.image_size, cfg.image_size)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.18, contrast=0.18, saturation=0.18, hue=0.03),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

infer_tfms = T.Compose([
    T.Resize((cfg.image_size, cfg.image_size)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = ColorDataset(train_balanced_df, transform=train_tfms)
val_ds = ColorDataset(val_df, transform=infer_tfms)
test_ds = ColorDataset(test_df, transform=infer_tfms)

# WeightedRandomSampler = oversampling редких классов на уровне выборки батчей
cls_counts = train_balanced_df['label'].value_counts().sort_index()
cls_weights = 1.0 / cls_counts.values.astype(np.float64)
sample_weights = train_balanced_df['label'].map({i: w for i, w in enumerate(cls_weights)}).values
sample_weights = torch.as_tensor(sample_weights, dtype=torch.double)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_balanced_df),
    replacement=True,
)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    sampler=sampler,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_ds,
    batch_size=cfg.eval_batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_ds,
    batch_size=cfg.eval_batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)


## Модель 1: ScratchResNet (ручная реализация)


In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)

        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + identity)
        return out


class ScratchResNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.in_ch = 32

        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )

        self.layer1 = self._make_layer(32, blocks=2, stride=1)
        self.layer2 = self._make_layer(64, blocks=2, stride=2)
        self.layer3 = self._make_layer(128, blocks=2, stride=2)
        self.layer4 = self._make_layer(256, blocks=2, stride=2)

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256, num_classes)

    def _make_layer(self, out_ch, blocks, stride):
        layers = [BasicBlock(self.in_ch, out_ch, stride=stride)]
        self.in_ch = out_ch
        for _ in range(1, blocks):
            layers.append(BasicBlock(self.in_ch, out_ch, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


## Общие функции обучения и валидации


In [ ]:
def evaluate_model(model, loader, criterion, show_tqdm=False, desc='eval', leave=False):
    model.eval()
    losses = []
    y_true, y_pred = [], []

    iterator = tqdm(loader, desc=desc, leave=leave, disable=not show_tqdm)

    with torch.no_grad():
        for xb, yb in iterator:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)

            losses.append(loss.item())
            y_true.extend(yb.cpu().numpy().tolist())
            y_pred.extend(logits.argmax(dim=1).cpu().numpy().tolist())

            if show_tqdm and losses:
                iterator.set_postfix(loss=f'{np.mean(losses):.4f}')

    f1 = f1_score(y_true, y_pred, average='macro')
    return float(np.mean(losses)), float(f1), y_true, y_pred


def train_model(model, model_name, train_loader, val_loader, epochs, lr, weight_decay, patience=3, use_tqdm=True, tqdm_leave=False):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)

    hist = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': []}
    best_state = None
    best_val_f1 = -1.0
    bad_epochs = 0

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        y_true_tr, y_pred_tr = [], []

        train_iter = tqdm(
            train_loader,
            desc=f'[{model_name}] train {epoch:02d}/{epochs}',
            leave=tqdm_leave,
            disable=not use_tqdm,
        )

        for xb, yb in train_iter:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            y_true_tr.extend(yb.detach().cpu().numpy().tolist())
            y_pred_tr.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())

            if use_tqdm and train_losses:
                train_iter.set_postfix(loss=f'{np.mean(train_losses):.4f}')

        tr_loss = float(np.mean(train_losses))
        tr_f1 = float(f1_score(y_true_tr, y_pred_tr, average='macro'))

        val_loss, val_f1, _, _ = evaluate_model(
            model,
            val_loader,
            criterion,
            show_tqdm=use_tqdm,
            desc=f'[{model_name}] val   {epoch:02d}/{epochs}',
            leave=tqdm_leave,
        )
        scheduler.step(val_f1)

        hist['train_loss'].append(tr_loss)
        hist['train_f1'].append(tr_f1)
        hist['val_loss'].append(val_loss)
        hist['val_f1'].append(val_f1)

        print(f'[{model_name}] epoch {epoch:02d}/{epochs} | train_loss={tr_loss:.4f} train_f1={tr_f1:.4f} | val_loss={val_loss:.4f} val_f1={val_f1:.4f}')

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            print(f'[{model_name}] early stopping')
            break

    model.load_state_dict(best_state)
    return model, hist, best_val_f1


def plot_history(hist, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(hist['train_loss'], label='train')
    axes[0].plot(hist['val_loss'], label='val')
    axes[0].set_title(f'{title}: Loss')
    axes[0].legend()

    axes[1].plot(hist['train_f1'], label='train')
    axes[1].plot(hist['val_f1'], label='val')
    axes[1].set_title(f'{title}: F1_macro')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


In [ ]:
num_classes = len(label2id)

scratch_model = ScratchResNet(num_classes=num_classes)
scratch_model, hist_scratch, best_val_scratch = train_model(
    model=scratch_model,
    model_name='ScratchResNet',
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=cfg.epochs_scratch,
    lr=cfg.lr_scratch,
    weight_decay=cfg.weight_decay,
    patience=3,
    use_tqdm=cfg.use_tqdm,
    tqdm_leave=cfg.tqdm_leave,
)

plot_history(hist_scratch, 'ScratchResNet')
print('Best val F1_macro (ScratchResNet):', round(best_val_scratch, 4))


## Модель 2: ResNet18 (ImageNet) + дообучение


In [ ]:
model_rn18 = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
model_rn18.fc = nn.Linear(model_rn18.fc.in_features, num_classes)

model_rn18, hist_rn18, best_val_rn18 = train_model(
    model=model_rn18,
    model_name='ResNet18-ImageNet',
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=cfg.epochs_pretrained_1,
    lr=cfg.lr_pretrained,
    weight_decay=cfg.weight_decay,
    patience=2,
    use_tqdm=cfg.use_tqdm,
    tqdm_leave=cfg.tqdm_leave,
)

plot_history(hist_rn18, 'ResNet18-ImageNet')
print('Best val F1_macro (ResNet18-ImageNet):', round(best_val_rn18, 4))


## Модель 3: MobileNetV3-Large (ImageNet) + дообучение


In [ ]:
model_mnv3 = tv_models.mobilenet_v3_large(weights=tv_models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
model_mnv3.classifier[-1] = nn.Linear(model_mnv3.classifier[-1].in_features, num_classes)

model_mnv3, hist_mnv3, best_val_mnv3 = train_model(
    model=model_mnv3,
    model_name='MobileNetV3-ImageNet',
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=cfg.epochs_pretrained_2,
    lr=cfg.lr_pretrained,
    weight_decay=cfg.weight_decay,
    patience=2,
    use_tqdm=cfg.use_tqdm,
    tqdm_leave=cfg.tqdm_leave,
)

plot_history(hist_mnv3, 'MobileNetV3-ImageNet')
print('Best val F1_macro (MobileNetV3-ImageNet):', round(best_val_mnv3, 4))


## Оценка на test и сравнение моделей


In [ ]:
criterion = nn.CrossEntropyLoss()

loss_s, f1_s, y_true_s, y_pred_s = evaluate_model(
    scratch_model,
    test_loader,
    criterion,
    show_tqdm=cfg.use_tqdm,
    desc='[ScratchResNet] test',
    leave=cfg.tqdm_leave,
)
loss_r, f1_r, y_true_r, y_pred_r = evaluate_model(
    model_rn18,
    test_loader,
    criterion,
    show_tqdm=cfg.use_tqdm,
    desc='[ResNet18] test',
    leave=cfg.tqdm_leave,
)
loss_m, f1_m, y_true_m, y_pred_m = evaluate_model(
    model_mnv3,
    test_loader,
    criterion,
    show_tqdm=cfg.use_tqdm,
    desc='[MobileNetV3] test',
    leave=cfg.tqdm_leave,
)

results = pd.DataFrame([
    {'model': 'ScratchResNet (from scratch)', 'test_loss': loss_s, 'test_f1_macro': f1_s},
    {'model': 'ResNet18 (ImageNet finetune)', 'test_loss': loss_r, 'test_f1_macro': f1_r},
    {'model': 'MobileNetV3-Large (ImageNet finetune)', 'test_loss': loss_m, 'test_f1_macro': f1_m},
]).sort_values('test_f1_macro', ascending=False).reset_index(drop=True)

results


In [ ]:
plt.figure(figsize=(8, 4))
if HAS_SEABORN:
    sns.barplot(data=results, x='model', y='test_f1_macro')
else:
    plt.bar(results['model'], results['test_f1_macro'])
plt.xticks(rotation=15, ha='right')
plt.ylim(0, 1)
plt.title('Сравнение F1_macro на test')
plt.tight_layout()
plt.show()


In [ ]:
best_model_name = results.iloc[0]['model']
print('Лучшая модель:', best_model_name)

if best_model_name.startswith('ScratchResNet'):
    y_true_best, y_pred_best = y_true_s, y_pred_s
elif best_model_name.startswith('ResNet18'):
    y_true_best, y_pred_best = y_true_r, y_pred_r
else:
    y_true_best, y_pred_best = y_true_m, y_pred_m

print('\nClassification report (лучшая модель):')
print(classification_report(y_true_best, y_pred_best, target_names=[id2label[i] for i in range(num_classes)]))


In [ ]:
cm = confusion_matrix(y_true_best, y_pred_best)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(9, 7))
labels = [id2label[i] for i in range(num_classes)]
if HAS_SEABORN:
    sns.heatmap(
        cm_norm,
        cmap='Blues',
        xticklabels=labels,
        yticklabels=labels,
    )
else:
    plt.imshow(cm_norm, cmap='Blues', aspect='auto')
    plt.colorbar()
    plt.xticks(range(len(labels)), labels, rotation=45, ha='right')
    plt.yticks(range(len(labels)), labels)

plt.title('Нормализованная confusion matrix (лучшая модель)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()


In [ ]:
threshold = 0.8
print('Итоговая таблица:')
print(results.to_string(index=False))

best_f1 = float(results.iloc[0]['test_f1_macro'])
print(f'\nЛучший F1_macro = {best_f1:.4f}')
print('Требование F1_macro > 0.8:', 'выполнено' if best_f1 > threshold else 'не выполнено')

print('\nОбоснование работы с дисбалансом:')
print('- удаляется только шумовой класс unlisted (если включено в Config);')
if cfg.merge_rare_into_other:
    print('- редкие классы объединены в other_rare:', prep_report['merged_rare_classes'])
else:
    print('- редкие классы не объединялись (сохраняем больше классов);')
print('- частые классы ограничены по количеству в train (undersampling);')
print('- для train используется WeightedRandomSampler (oversampling редких в батчах).')
